<a href="https://colab.research.google.com/github/aryanshenoy02/Banking-Fraud-Detection/blob/main/notebooks/isolation-forest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import pandas as pd

try:
    # Skipping the first 3 rows of metadata to start directly from the table headers
    df = pd.read_csv('AryanTransactions.csv', skiprows=3)
    print("Successfully loaded by skipping the metadata rows:")
    display(df.head(10))
except Exception as e:
    print(f"Error reading CSV: {e}")

In [ ]:
combined = df['Date'].astype(str) + ' ' + df['Time'].astype(str)

parsed = pd.to_datetime(
    combined.str.replace(r'\bSept\b', 'Sep', regex=True),
    format='%b %d, %Y %I:%M %p',
    errors='coerce'
)

invalid_rows = df[parsed.isna()]

display(invalid_rows)
df = df[parsed.notna()].copy()

df['Datetime'] = parsed[parsed.notna()]

In [ ]:
df['Date'] = df['Date'].str.replace(
    r'\bSept\b', 'Sep', regex=True
)

df['Datetime'] = pd.to_datetime(
    df['Date'] + ' ' + df['Time'],
    format='%b %d, %Y %I:%M %p'
)

In [ ]:
import numpy as np

# 1. Drop rows where all columns are NaN (if any exist at the end of the file)
df = df.dropna(how='all').copy()

# 2. Clean the 'Transaction Details' column
# We remove case-insensitive variations of "Paid to " and "Received from "
df['Transaction Details'] = df['Transaction Details'].str.replace(r'^(Paid to|Received from)\s+', '', case=False, regex=True)

# 3. Create a combined datetime column to sort chronologically and compute differences
df['Datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], format='%b %d, %Y %I:%M %p')

# Sort chronologically so we can calculate the time difference forward in time
df = df.sort_values('Datetime').reset_index(drop=True)

# 4. Calculate the time difference between consecutive transactions (in hours)
df['Time_Diff_Hours'] = df['Datetime'].diff().dt.total_seconds() / 3600.0

# Display the cleaned and processed DataFrame
display(df[['Date', 'Time', 'Transaction Details', 'Transaction Type', 'Amount', 'Time_Diff_Hours']].head(10))

In [ ]:
# sort to datatime
df = df.sort_values('Datetime').reset_index(drop=True)

# Calculate cumulative transaction count per counterparty (excluding current transaction)
df['Counterparty_Transaction_Count'] = df.groupby('Transaction Details').cumcount()

# Display the results emphasizing the new column
display(df[['Datetime', 'Transaction Details', 'Counterparty_Transaction_Count']].head(20))

In [ ]:
import numpy as np

df['Log_Amount'] = np.log1p(df['Amount'])
df = df.drop(columns = ['Transaction ID', 'UTR', 'Date', 'Time'])
df.head()

In [ ]:
df[['Credit/debit instrument']]

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest

df['hour'] = df['Datetime'].dt.hour
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
df['dayofweek'] = df['Datetime'].dt.dayofweek
df['is_new_counterparty'] = (df['Counterparty_Transaction_Count'] == 0).astype(int)
df['is_debit'] = (df['Transaction Type'] == 'DEBIT').astype(int)
#df = pd.get_dummies(df, columns=['Credit/debit instrument'], dtype=int)

# 2. Split by date
cut = int(len(df) * 0.75)
train = df.iloc[:cut].copy()
test = df.iloc[cut:].copy()

# 3. Choose features, fill NaN using TRAIN only
#inst_cols = [c for c in df.columns if c.startswith('Credit/debit instrument_')]
features = ['Log_Amount', 'Time_Diff_Hours', 'Counterparty_Transaction_Count',
            'hour_sin', 'hour_cos', 'dayofweek',
            'is_debit']

#for the 1st value since its NaN
median_gap = train['Time_Diff_Hours'].median()
train['Time_Diff_Hours'] = train['Time_Diff_Hours'].fillna(median_gap)
test['Time_Diff_Hours'] = test['Time_Diff_Hours'].fillna(median_gap)

# 4. Train and score
model = IsolationForest(n_estimators=200, random_state=42)
model.fit(train[features])

train_scores = model.decision_function(train[features])
test['score'] = model.decision_function(test[features])

# 5. Threshold and flag
threshold = np.percentile(train_scores, 1)
test['flagged'] = test['score'] < threshold

print(test['flagged'].mean())
print(test.sort_values('score').head(10)[['Transaction Details', 'Amount', 'Datetime', 'score']])

In [ ]:
# 1. Flip so higher = more suspicious, then scale using TRAIN stats
train_raw = -model.decision_function(train[features])
test_raw = -model.decision_function(test[features])
lo, hi = train_raw.min(), train_raw.max()
train['anomaly'] = np.clip((train_raw - lo) / (hi - lo), 0, None)
test['anomaly'] = np.clip((test_raw - lo) / (hi - lo), 0, None)

# 2. Familiarity discount
def trust_divisor(count, k=5):
    return 1 + count / (count + k)

train['final'] = train['anomaly'] / trust_divisor(train['Counterparty_Transaction_Count'])
test['final'] = test['anomaly'] / trust_divisor(test['Counterparty_Transaction_Count'])

# 3. Threshold from train, flag test
threshold = np.percentile(train['final'], 99)
test['flagged'] = test['final'] > threshold

print(test['flagged'].mean())
print(test.sort_values('final', ascending=False).head(10)
      [['Transaction Details', 'Amount', 'Datetime', 'final']])

In [ ]:
df.head(10)